# 03 · Ejercicios — Diseños $2^k$ completos
(Python)

**Semana 3 — Diseños $2^k$ y fraccionados.**

**Objetivos**
- Analizar diseños $2^3$ con réplicas mediante ANOVA completo e identificar efectos significativos.
- Usar la gráfica de probabilidad normal de efectos en diseños $2^4$ sin réplicas.
- Ajustar modelos reducidos y encontrar las condiciones óptimas del proceso.
- Validar supuestos de normalidad y varianza constante.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels`

> Teoría: [../teoria/01-disenos-2k.md](../teoria/01-disenos-2k.md)  
> Equivalente en R: [03-ejercicios-2k_r.ipynb](03-ejercicios-2k_r.ipynb)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols
from statsmodels.graphics.factorplots import interaction_plot

---
## Ejercicio 1 · Rendimiento de reacción química ($2^3$ con réplicas)

Un ingeniero estudia el **rendimiento (%)** de una reacción química en función de tres
factores a dos niveles con dos réplicas completas (16 corridas):

| Factor | $-1$ | $+1$ |
|--------|------|------|
| `temperatura` | 70 °C | 90 °C |
| `tiempo` | 30 min | 60 min |
| `catalizador` | 1 % | 2 % |

**Objetivo:** identificar efectos significativos y condiciones de máximo rendimiento.

In [ ]:
df1 = pd.read_csv('../datos/reaccion-quimica-2k.csv')
print(df1.shape, '\n')
print('Medias por combinacion principal:')
print(df1.pivot_table('rendimiento', index=['temperatura','catalizador'],
                      columns='tiempo').round(1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ['temperatura', 'tiempo', 'catalizador']):
    df1.boxplot(column='rendimiento', by=col, ax=ax, grid=False,
                boxprops=dict(color='steelblue'))
    ax.set_title(f'Por {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Rendimiento (%)')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
# Modelo factorial completo 2^3
modelo1 = ols('rendimiento ~ temperatura*tiempo*catalizador', data=df1).fit()
anova1 = sm.stats.anova_lm(modelo1, typ=2)
print('ANOVA completo (Tipo II):')
print(anova1.round(3))

In [ ]:
# Grafica de interaccion temperatura x tiempo (efectos mas grandes)
fig, ax = plt.subplots(figsize=(7, 5))
interaction_plot(df1['tiempo'], df1['temperatura'], df1['rendimiento'],
                 ax=ax, xlabel='Tiempo', ylabel='Rendimiento medio (%)',
                 legendtitle='Temperatura', ms=8)
ax.set_title('Interaccion Temperatura x Tiempo')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Modelo reducido: solo terminos significativos
modelo1r = ols('rendimiento ~ temperatura + tiempo + catalizador + temperatura:tiempo',
               data=df1).fit()
anova1r = sm.stats.anova_lm(modelo1r, typ=2)
print('Modelo reducido:')
print(anova1r.round(3))
print(f'\nR2 = {modelo1r.rsquared:.4f}')

resid1 = modelo1r.resid
ajust1 = modelo1r.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(resid1, line='s', ax=axes[0])
axes[0].set_title('Q-Q de residuales')
axes[1].scatter(ajust1, resid1, alpha=0.8)
axes[1].axhline(0, color='gray', ls='--')
axes[1].set_xlabel('Valores ajustados')
axes[1].set_ylabel('Residuales')
axes[1].set_title('Residuales vs. ajustados')
plt.tight_layout()
plt.show()
print('Shapiro-Wilk:', stats.shapiro(resid1))

---
## Ejercicio 2 · Resistencia del papel ($2^4$ sin réplicas)

Se estudia la **resistencia a la traccion (N/m)** del papel en funcion de 4 factores,
con una sola replicacion (16 corridas). Sin réplicas no hay grados de libertad para
el error: se usa la **grafica de probabilidad normal de efectos** (metodo de Daniel).

| Factor | $-1$ | $+1$ |
|--------|------|------|
| `temperatura` (A) | 60 °C | 80 °C |
| `humedad` (B) | 40 % | 60 % |
| `velocidad` (C) | 100 m/min | 150 m/min |
| `presion` (D) | 2 bar | 4 bar |

In [ ]:
df2 = pd.read_csv('../datos/papel-resistencia-2k.csv')
print(df2.head(8).to_string(index=False))

# Ajustar modelo saturado (todos los terminos)
modelo2 = ols('resistencia ~ temperatura*humedad*velocidad*presion', data=df2).fit()

# Efectos = 2 * coeficiente de regresion
efectos2 = modelo2.params.drop('Intercept') * 2
alias = {
    'temperatura':'A', 'humedad':'B', 'velocidad':'C', 'presion':'D',
    'temperatura:humedad':'AB', 'temperatura:velocidad':'AC',
    'temperatura:presion':'AD', 'humedad:velocidad':'BC',
    'humedad:presion':'BD', 'velocidad:presion':'CD',
    'temperatura:humedad:velocidad':'ABC', 'temperatura:humedad:presion':'ABD',
    'temperatura:velocidad:presion':'ACD', 'humedad:velocidad:presion':'BCD',
    'temperatura:humedad:velocidad:presion':'ABCD'
}
efectos2.index = [alias.get(x, x) for x in efectos2.index]
print('\nEfectos estimados (ordenados por magnitud):')
print(efectos2.abs().sort_values(ascending=False).round(2))

In [ ]:
# Grafica de probabilidad normal de efectos (Daniel)
eff_sorted = efectos2.sort_values()
n = len(eff_sorted)
probs = [(i + 0.5) / n for i in range(n)]
z_scores = stats.norm.ppf(probs)

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(z_scores, eff_sorted.values, s=65, color='steelblue', zorder=3)
for z, e, name in zip(z_scores, eff_sorted.values, eff_sorted.index):
    ax.annotate(name, (z, e), xytext=(5, 2), textcoords='offset points', fontsize=9)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Cuantiles normales teoricos')
ax.set_ylabel('Efecto estimado')
ax.set_title('Grafica normal de efectos — Papel resistencia (Daniel)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Efectos activos identificados: A(temperatura), B(humedad), D(presion), AD
modelo2r = ols('resistencia ~ temperatura + humedad + presion + temperatura:presion',
               data=df2).fit()
anova2r = sm.stats.anova_lm(modelo2r, typ=2)
print('Modelo reducido (A, B, D, AD):')
print(anova2r.round(3))
print(f'\nR2 = {modelo2r.rsquared:.4f}')

# Diagnosticos
resid2 = modelo2r.resid
ajust2 = modelo2r.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(resid2, line='s', ax=axes[0])
axes[0].set_title('Q-Q residuales - papel')
axes[1].scatter(ajust2, resid2, alpha=0.8)
axes[1].axhline(0, color='gray', ls='--')
axes[1].set_xlabel('Ajustados')
axes[1].set_ylabel('Residuales')
axes[1].set_title('Residuales vs. ajustados - papel')
plt.tight_layout()
plt.show()
print('Shapiro-Wilk:', stats.shapiro(resid2))

# Optimo: A=+1 (alta temp), B=-1 (baja humedad), D=+1 (alta presion)
pred_opt = modelo2r.predict(pd.DataFrame({'temperatura':[1],'humedad':[-1],'presion':[1]}))
print(f'\nPrediccion optima (A=+1, B=-1, D=+1): {pred_opt.values[0]:.1f} N/m')

---
## Ejercicio 3 · Grabado de semiconductores ($2^3$ con réplicas)

Se estudia la **tasa de grabado (Å/min)** en un proceso de grabado de obleas de
silicio. Tres factores a dos niveles, dos réplicas (16 corridas):

| Factor | $-1$ | $+1$ |
|--------|------|------|
| `potencia` (A) | 100 W | 150 W |
| `presion_gas` (B) | 0.8 Torr | 1.2 Torr |
| `temperatura` (C) | 15 °C | 25 °C |

**Objetivo:** identificar los efectos dominantes y la combinacion que maximiza la tasa de grabado.

In [ ]:
df3 = pd.read_csv('../datos/microelectronica-2k.csv')
print('Medias por celda:')
print(df3.pivot_table('tasa_grabado', index=['potencia','temperatura'],
                      columns='presion_gas').round(1))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col in zip(axes, ['potencia', 'presion_gas', 'temperatura']):
    df3.boxplot(column='tasa_grabado', by=col, ax=ax, grid=False,
                boxprops=dict(color='darkgreen'))
    ax.set_title(f'Por {col}')
    ax.set_ylabel('Tasa grabado (A/min)')
plt.suptitle('')
plt.tight_layout()
plt.show()

In [ ]:
modelo3 = ols('tasa_grabado ~ potencia*presion_gas*temperatura', data=df3).fit()
anova3 = sm.stats.anova_lm(modelo3, typ=2)
print('ANOVA completo 2^3 con replicas:')
print(anova3.round(2))

In [ ]:
# Interaccion potencia x temperatura (la mas importante)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
interaction_plot(df3['temperatura'], df3['potencia'], df3['tasa_grabado'],
                 ax=axes[0], xlabel='Temperatura', ylabel='Tasa grabado media',
                 legendtitle='Potencia', ms=8)
axes[0].set_title('Interaccion Potencia x Temperatura')
axes[0].grid(True, alpha=0.3)

interaction_plot(df3['potencia'], df3['presion_gas'], df3['tasa_grabado'],
                 ax=axes[1], xlabel='Potencia', ylabel='Tasa grabado media',
                 legendtitle='Presion gas', ms=8)
axes[1].set_title('Interaccion Potencia x Presion gas')
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Condiciones optimas
pred_celdas = df3.groupby(['potencia','presion_gas','temperatura'])['tasa_grabado'].mean()
print('\nMedia mas alta:')
print(pred_celdas.sort_values(ascending=False).head(3))

In [ ]:
# Modelo reducido: efectos significativos A, B, C, AC
modelo3r = ols('tasa_grabado ~ potencia + presion_gas + temperatura + potencia:temperatura',
               data=df3).fit()
print(f'R2 modelo reducido = {modelo3r.rsquared:.4f}')

resid3 = modelo3r.resid
ajust3 = modelo3r.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(resid3, line='s', ax=axes[0])
axes[0].set_title('Q-Q residuales - semiconductores')
axes[1].scatter(ajust3, resid3, alpha=0.8)
axes[1].axhline(0, color='gray', ls='--')
axes[1].set_xlabel('Ajustados')
axes[1].set_ylabel('Residuales')
axes[1].set_title('Residuales vs. ajustados - semiconductores')
plt.tight_layout()
plt.show()
print('Shapiro-Wilk:', stats.shapiro(resid3))

---
## Ejercicio 4 · Densidad de espuma de poliuretano ($2^4$ sin réplicas)

Un formulador quimico estudia la **densidad (kg/m³)** de espuma de poliuretano.
Cuatro factores, sin réplicas (16 corridas):

| Factor | $-1$ | $+1$ |
|--------|------|------|
| `isocianato` (A) | 45 PHR | 55 PHR |
| `poliol` (B) | 100 PHR | 110 PHR |
| `temperatura` (C) | 20 °C | 30 °C |
| `humedad` (D) | 40 % | 60 % |

In [ ]:
df4 = pd.read_csv('../datos/espuma-poliuretano-2k.csv')

modelo4 = ols('densidad ~ isocianato*poliol*temperatura*humedad', data=df4).fit()
efectos4 = modelo4.params.drop('Intercept') * 2
alias4 = {
    'isocianato':'A','poliol':'B','temperatura':'C','humedad':'D',
    'isocianato:poliol':'AB','isocianato:temperatura':'AC',
    'isocianato:humedad':'AD','poliol:temperatura':'BC',
    'poliol:humedad':'BD','temperatura:humedad':'CD',
    'isocianato:poliol:temperatura':'ABC','isocianato:poliol:humedad':'ABD',
    'isocianato:temperatura:humedad':'ACD','poliol:temperatura:humedad':'BCD',
    'isocianato:poliol:temperatura:humedad':'ABCD'
}
efectos4.index = [alias4.get(x, x) for x in efectos4.index]
print('Efectos estimados (por magnitud):')
print(efectos4.abs().sort_values(ascending=False).round(2))

In [ ]:
# Grafica de Daniel
eff_sorted4 = efectos4.sort_values()
n4 = len(eff_sorted4)
z4 = stats.norm.ppf([(i + 0.5) / n4 for i in range(n4)])

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(z4, eff_sorted4.values, s=65, color='darkorange', zorder=3)
for z, e, name in zip(z4, eff_sorted4.values, eff_sorted4.index):
    ax.annotate(name, (z, e), xytext=(5, 2), textcoords='offset points', fontsize=9)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Cuantiles normales teoricos')
ax.set_ylabel('Efecto estimado')
ax.set_title('Grafica normal de efectos — Espuma poliuretano (Daniel)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Modelo reducido: efectos activos A, B, C, D, AC
modelo4r = ols('densidad ~ isocianato + poliol + temperatura + humedad + isocianato:temperatura',
               data=df4).fit()
anova4r = sm.stats.anova_lm(modelo4r, typ=2)
print('Modelo reducido (A, B, C, D, AC):')
print(anova4r.round(3))
print(f'\nR2 = {modelo4r.rsquared:.4f}')

resid4 = modelo4r.resid
ajust4 = modelo4r.fittedvalues
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sm.qqplot(resid4, line='s', ax=axes[0])
axes[0].set_title('Q-Q residuales - espuma')
axes[1].scatter(ajust4, resid4, alpha=0.8)
axes[1].axhline(0, color='gray', ls='--')
axes[1].set_xlabel('Ajustados')
axes[1].set_ylabel('Residuales')
axes[1].set_title('Residuales vs. ajustados - espuma')
plt.tight_layout()
plt.show()
print('Shapiro-Wilk:', stats.shapiro(resid4))

# Predecir en las condiciones activas extremas
combinaciones = pd.DataFrame({
    'isocianato':[1,1,-1,-1],'poliol':[-1,-1,-1,-1],
    'temperatura':[-1,-1,1,1],'humedad':[1,-1,1,-1]
})
combinaciones['pred'] = modelo4r.predict(combinaciones)
print('\nPredicciones en combinaciones extremas:')
print(combinaciones.round(1))